In [2]:
import numpy as np
import pandas as pd

# 1. Dataset split

def split_tree(tree, criteria, threshold):
    left_tree = tree[tree[criteria] < threshold]
    right_tree = tree[tree[criteria] >= threshold]
    return left_tree, right_tree


# 2. Node entropy (binary target)

def entropy_node(tree, binary_target):
    '''Entropy measure the uncertainty (impurety) of a node. 
    The higher the entropy, the more uncertain the node is.
    H(0.5) = 1 (most uncertain
    H(0) = H(1) = 0 (most certain)
    '''
    n = tree.shape[0]
    
    if n == 0:
        return 0
    
    p = tree[binary_target].mean()
    
    if p == 0 or p == 1:
        return 0
    
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


# 3. Entropy after split

def entropy_split(left_tree, right_tree, binary_target):
    '''Entropy after split'''

    n_left = left_tree.shape[0]
    n_right = right_tree.shape[0]
    n_total = n_left + n_right
    
    if n_total == 0:
        return 0
    
    weight_left = n_left / n_total
    weight_right = n_right / n_total

    weighted_entropy = (
        weight_left * entropy_node(left_tree, binary_target) 
        + weight_right * entropy_node(right_tree, binary_target)
    )
    
    return weighted_entropy


# 4. Information gain

def information_gain(tree, criteria, threshold, binary_target):
    '''Maximize the decrease in entropy after the split'''

    parent_entropy = entropy_node(tree, binary_target)

    left_tree, right_tree = split_tree(tree, criteria, threshold)
    split_entropy = entropy_split(left_tree, right_tree, binary_target)

    gain = parent_entropy - split_entropy

    return gain


# 5. Find the best split

def find_best_threshold(tree, criteria, binary_target):
    '''Find the threshold that maximizes the information gain.
    Simple approach that iterates through all unique values of the criteria 
    and calculates the gain for each threshold. 
    '''
    thresholds = tree[criteria].unique()
    best_gain = -np.inf
    best_threshold = None

    for threshold in thresholds:
        gain = information_gain(tree, criteria, threshold, binary_target)
        if gain > best_gain:
            best_gain = gain
            best_threshold = threshold

    return best_threshold, best_gain

In [3]:
# Load the dataset

import os

data_path = '/Users/davidtbo/Library/Mobile Documents/com~apple~CloudDocs/data/external'
file_path = os.path.join(data_path, 'diabetes.csv')

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Fichier introuvable : {file_path}")

df = pd.read_csv(file_path)

# Standardize column names
df.columns = df.columns.str.lower()

# Drop duplicates
df = df.drop_duplicates()

# # Quick diagnostic
# print(df.info())
# print(df.isna().mean())

df.head()

,pregnancies,glucose,bloodpressure,skinthickness,insulin,bmi,diabetespedigreefunction,age,outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
# 0. Train-test split

X = df.drop(columns='outcome')
y = df['outcome']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
df_train = pd.concat([X_train, y_train], axis=1)

In [6]:
df_train.head()

,pregnancies,glucose,bloodpressure,skinthickness,insulin,bmi,diabetespedigreefunction,age,outcome
353,1,90,62,12,43,27.2,0.580,24,0
711,5,126,78,27,22,29.6,0.439,40,0
373,2,105,58,40,94,34.9,0.225,25,0
46,1,146,56,0,0,29.7,0.564,29,0
682,0,95,64,39,105,44.6,0.366,22,0


In [7]:
# Test the function on the training set
find_best_threshold(df_train, "glucose", "outcome")

(np.int64(155), np.float64(0.14364017033135035))

In [8]:
# Find the best threshold for each feature
for col in df_train.columns[:-1]:  # Exclude the target column
    best_threshold, best_gain = find_best_threshold(df_train, col, "outcome")
    print(f"Best threshold for {col}: {best_threshold} with gain: {best_gain:.4f}")

Best threshold for pregnancies: 7 with gain: 0.0388
Best threshold for glucose: 155 with gain: 0.1436
Best threshold for bloodpressure: 70 with gain: 0.0185
Best threshold for skinthickness: 33 with gain: 0.0229
Best threshold for insulin: 129 with gain: 0.0368
Best threshold for bmi: 29.7 with gain: 0.0741
Best threshold for diabetespedigreefunction: 0.528 with gain: 0.0243
Best threshold for age: 28 with gain: 0.0654


In [9]:
# The best gain is for glucose at threshold 155. Let's split the tree on this threshold and calculate the entropy after the split.
left_tree, right_tree = split_tree(df_train, "glucose", 155)

Note: We can improve the previous loop:

In [20]:
# Find the best threshold for each feature

best_gain = -np.inf
best_threshold = None

for col in df_train.columns[:-1]:  # Exclude the target column
    threshold, gain = find_best_threshold(df_train, col, "outcome")
    if gain > best_gain:
        best_gain = gain
        best_threshold = threshold
print(f"Best threshold: {best_threshold} with gain: {best_gain:.4f}")

Best threshold: 155 with gain: 0.1436


In [10]:
# And we can find the best threshold for each feature in the left tree (glucose < 155)
for col in left_tree.columns[:-1]:  # Exclude the target column
    best_threshold, best_gain = find_best_threshold(left_tree, col, "outcome")
    print(f"Best threshold for {col}: {best_threshold} with gain: {best_gain:.4f}")

Best threshold for pregnancies: 7 with gain: 0.0270
Best threshold for glucose: 100 with gain: 0.0539
Best threshold for bloodpressure: 80 with gain: 0.0152
Best threshold for skinthickness: 31 with gain: 0.0205
Best threshold for insulin: 122 with gain: 0.0179
Best threshold for bmi: 26.8 with gain: 0.0710
Best threshold for diabetespedigreefunction: 0.51 with gain: 0.0233
Best threshold for age: 29 with gain: 0.0534


In [15]:
# For the left tree, the best gain is for bmi at threshold 26.8. 
# Let's split the tree on this threshold and calculate the entropy after the split.

left_subtree, right_subtree = split_tree(left_tree, "bmi", 26.8)

1. This is a manual process (we decide left and right), and we could continue to split till the last point.right_tree. 

So we need to make this process automated and give to it a stopping criterion (e.g., max depth, min samples per leaf, etc.)

2. **Let's build a recursive function to build the decision tree.**  

We will use a simple stopping criterion: if the gain is less than a certain threshold, we will stop splitting.

__build_tree(data)__  
* if data is pure enough or too small:  
    * return a leaf  

* find the best split
* split data into left_data and right_data  

* build the sub left tree with left_data  
* build the sub right tree with right_data  

* return a node containing:  
    * variable  
    * threshold  
    * left_subtree  
    * right_subtree  

The key point is:

* **left_subtree = build_tree(left_data)**

* **right_subtree = build_tree(right_data)**

The function calls itself with smaller datasets.  

3. We need a structure (dictionary) to store:

* The tree

In [16]:
node = {
    "type": "node",
    "feature": "glucose",
    "threshold": 120,
    "left": left_subtree,
    "right": right_subtree,
}

* And a leaf

In [17]:
leaf = {
    "type": "leaf",
    "prediction": 1
}

4. We need to define a **stopping criterion**.  

It ensures that the tree does not grow indefinitely.

Simple examples:

* if entropy_node(tree, target) == 0:
    * return leaf

* if depth >= max_depth:
    * return leaf

* if len(tree) < min_samples_split:
    * return leaf

* if best_gain <= 0:
    * return leaf

The prediction of a leaf is often the majority class of the samples in that leaf.  

In this case, since we are working with a binary classification problem, the prediction would be 1:

>prediction = int(tree[target].mean() >= 0.5)

5. **Skeleton version**

In [ ]:
def majority_class(tree, binary_target):
    return int(tree[binary_target].mean() >= 0.5)


def build_tree(tree, features, binary_target, depth=0, max_depth=3, min_samples_split=5):
    if (
        entropy_node(tree, binary_target) == 0
        or depth >= max_depth 
        or len(tree) < min_samples_split
        ):
        return {
            "type": "leaf",
            "prediction": majority_class(tree, binary_target)
        }

    best_feature = None
    best_threshold = None
    best_gain = -np.inf

    for feature in features:
        threshold, gain = find_best_threshold(tree, feature, binary_target)

        if gain > best_gain:
            best_gain = gain
            best_feature = feature
            best_threshold = threshold

    if best_gain <= 0:
        return {
            "type": "leaf",
            "prediction": majority_class(tree, binary_target)
        }

    left_tree, right_tree = split_tree(tree, best_feature, best_threshold)

    return {
        "type": "node",
        "feature": best_feature,
        "threshold": best_threshold,
        "gain": best_gain,
        "left": build_tree(
            left_tree,
            features,
            binary_target,
            depth=depth + 1,
            max_depth=max_depth,
            min_samples_split=min_samples_split
        ),
        "right": build_tree(
            right_tree,
            features,
            binary_target,
            depth=depth + 1,
            max_depth=max_depth,
            min_samples_split=min_samples_split
        )
    }

6. Intuitive lecture. 

When you call:

> tree_model = build_tree(df_train, features, "target", max_depth=3)

Python does:

build_tree(racine)
* build_tree(sous-table gauche)
    * build_tree(sous-table gauche-gauche)
    * build_tree(sous-table gauche-droite)

* build_tree(sous-table droite)
    * build_tree(sous-table droite-gauche)
    * build_tree(sous-table droite-droite)

This is the manual process, automotized.

7. Then, prediction

Once the tree has been built, predire revient à descendre dans le dictionnaire:

In [ ]:
def predict_one(row, tree_model):
    if tree_model["type"] == "leaf":
        return tree_model["prediction"]

    feature = tree_model["feature"]
    threshold = tree_model["threshold"]

    if row[feature] < threshold:
        return predict_one(row, tree_model["left"])
    else:
        return predict_one(row, tree_model["right"])

Là encore, c’est récursif : tant qu’on n’est pas sur une feuille, on descend à gauche ou à droite.  

D'ailleurs on commence aussi par un test d'arrêt si feuille.

Petit détail : dans ton fichier actuel, tu charges load_breast_cancer(), pas diabetes. Et il faudra convertir X_train, y_train en DataFrame avec noms de colonnes + colonne target pour utiliser tes fonctions actuelles basées sur tree[criteria].

**Concrete case**

Cas concret. L’idée est :

* Tu concatènes X_train et y_train.
* Tu construis l’arbre avec build_tree(...).
* Pour prédire une ligne, predict_one(...) descend dans l’arbre jusqu’à une feuille.

Exemple avec un DataFrame df_train qui contient outcome :

In [24]:
# If X_train is already a DataFrame
df_train = X_train.copy()
df_train["outcome"] = y_train

features = [col for col in df_train.columns if col != "outcome"]

tree_model = build_tree(
    tree=df_train,
    features=features,
    binary_target="outcome",
    max_depth=3,
    min_samples_split=10
)

In [25]:
tree_model

{'type': 'node',
 'feature': 'glucose',
 'threshold': np.int64(155),
 'gain': np.float64(0.14364017033135035),
 'left': {'type': 'node',
  'feature': 'bmi',
  'threshold': np.float64(26.8),
  'gain': np.float64(0.07096267752687802),
  'left': {'type': 'node',
   'feature': 'pregnancies',
   'threshold': np.int64(3),
   'gain': np.float64(0.04995069273504149),
   'left': {'type': 'leaf', 'prediction': 0},
   'right': {'type': 'leaf', 'prediction': 0}},
  'right': {'type': 'node',
   'feature': 'age',
   'threshold': np.int64(31),
   'gain': np.float64(0.05482507314116902),
   'left': {'type': 'leaf', 'prediction': 0},
   'right': {'type': 'leaf', 'prediction': 0}}},
 'right': {'type': 'node',
  'feature': 'bmi',
  'threshold': np.float64(23.3),
  'gain': np.float64(0.05472453135895383),
  'left': {'type': 'leaf', 'prediction': 0},
  'right': {'type': 'node',
   'feature': 'pregnancies',
   'threshold': np.int64(7),
   'gain': np.float64(0.04551611789551313),
   'left': {'type': 'leaf', 

Prediction on row test data:

In [30]:
tree_model['feature']

'glucose'

In [31]:
row = X_test.iloc[0]
row

pregnancies                   7.000
glucose                     159.000
bloodpressure                64.000
skinthickness                 0.000
insulin                       0.000
bmi                          27.400
diabetespedigreefunction      0.294
age                          40.000
Name: 44, dtype: float64

In [33]:
prediction = predict_one(row, tree_model)
prediction

1

Si l’arbre dit :

glucose < 155 ?

Alors :

* oui => aller dans left
* non => aller dans right

Puis il recommence sur le sous-arbre suivant, par exemple :

bmi < 31.4 ?

Et dès qu’il tombe sur :

In [34]:
{"type": "leaf", "prediction": 1}

{'type': 'leaf', 'prediction': 1}

il retourne 1.

Pour prédire tout X_test :

In [35]:
y_pred = X_test.apply(lambda row: predict_one(row, tree_model), axis=1)

In [36]:
accuracy = (y_pred == y_test).mean()
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.6948


# END